In [ ]:
import io
import zipfile
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving dataset_link_phishing.csv (1).zip to dataset_link_phishing.csv (1).zip


In [ ]:
import io
import zipfile
import pandas as pd
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("Please select your dataset ZIP or CSV.")

filename = next(iter(uploaded))
file_data = io.BytesIO(uploaded[filename])

if filename.lower().endswith(".zip"):
    with zipfile.ZipFile(file_data) as archive:
        csv_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        if len(csv_files) != 1:
            raise ValueError("The ZIP must contain exactly one CSV.")

        df = pd.read_csv(
            archive.open(csv_files[0]),
            low_memory=False
        )
else:
    df = pd.read_csv(file_data, low_memory=False)

print("Original dataset shape:", df.shape)
print("\nOriginal label counts:")
print(df["status"].value_counts())

Saving dataset_link_phishing.csv (1).zip to dataset_link_phishing.csv (1) (1).zip
Original dataset shape: (19431, 87)

Original label counts:
status
legitimate    9716
phishing      9715
Name: count, dtype: int64


In [ ]:
data = df[["url", "status"]].copy()
data = data.dropna(subset=["url", "status"])

data["url"] = data["url"].astype(str).str.strip()
data["status"] = data["status"].astype(str).str.strip().str.lower()

data = data[data["url"].ne("")].copy()

expected_labels = {"legitimate", "phishing"}
unexpected = set(data["status"]) - expected_labels

if unexpected:
    raise ValueError(f"Unexpected labels found: {unexpected}")

label_counts = data.groupby("url")["status"].nunique()
conflicting_urls = label_counts[label_counts > 1].index

print("URLs with conflicting labels:", len(conflicting_urls))

data = data[~data["url"].isin(conflicting_urls)].copy()

data = data.drop_duplicates(subset="url").reset_index(drop=True)

data["label"] = data["status"].map({
    "legitimate": 0,
    "phishing": 1
})

print("\nUnique URLs:", len(data))
print("\nCleaned label counts:")
print(data["status"].value_counts())

URLs with conflicting labels: 0

Unique URLs: 11429

Cleaned label counts:
status
legitimate    5715
phishing      5714
Name: count, dtype: int64


In [ ]:
from urllib.parse import urlsplit
import ipaddress

def extract_url_features(url):
    url = str(url).strip()

    parse_url = url if "://" in url else "//" + url

    try:
        parts = urlsplit(parse_url)
        host = (parts.hostname or "").lower()
        path = parts.path
        query = parts.query
        scheme = parts.scheme.lower()
    except ValueError:
        host, path, query, scheme = "", "", "", ""

    try:
        ipaddress.ip_address(host)
        host_is_ip = 1
    except ValueError:
        host_is_ip = 0

    features = {
        "url_length": len(url),
        "hostname_length": len(host),
        "path_length": len(path),
        "query_length": len(query),
        "hostname_dot_count": host.count("."),
        "hostname_hyphen_count": host.count("-"),
        "digit_ratio": sum(c.isdigit() for c in url) / max(len(url), 1),
        "hostname_digit_ratio": (
            sum(c.isdigit() for c in host) / max(len(host), 1)
        ),
        "host_is_ip": host_is_ip,
        "uses_https": int(scheme == "https"),
        "has_punycode": int("xn--" in host),
        "has_hostname": int(bool(host)),
    }

    characters = {
        "dots": ".",
        "hyphens": "-",
        "at_signs": "@",
        "question_marks": "?",
        "ampersands": "&",
        "equals": "=",
        "underscores": "_",
        "percent_signs": "%",
        "slashes": "/",
    }

    for name, character in characters.items():
        features[f"count_{name}"] = url.count(character)

    return features

X = pd.DataFrame(
    data["url"].apply(extract_url_features).tolist(),
    index=data.index
)

y = data["label"].copy()

print("Feature table shape:", X.shape)
print("Missing feature values:", X.isna().sum().sum())
display(X.head())

Feature table shape: (11429, 21)
Missing feature values: 0


,url_length,hostname_length,path_length,query_length,hostname_dot_count,hostname_hyphen_count,digit_ratio,hostname_digit_ratio,host_is_ip,uses_https,...,has_hostname,count_dots,count_hyphens,count_at_signs,count_question_marks,count_ampersands,count_equals,count_underscores,count_percent_signs,count_slashes
0,46,20,10,8,2,0,0.108696,0.000000,0,0,...,1,3,0,0,1,0,1,0,0,3
1,128,120,1,0,10,0,0.054688,0.058333,0,0,...,1,10,0,0,0,0,0,0,0,3
2,52,25,20,0,2,0,0.000000,0.000000,0,0,...,1,3,0,0,0,0,0,0,0,4
3,21,13,1,0,2,0,0.142857,0.230769,0,0,...,1,2,0,0,0,0,0,0,0,3
4,28,19,1,0,2,0,0.000000,0.000000,0,1,...,1,2,0,0,0,0,0,0,0,3


In [ ]:
from sklearn.model_selection import train_test_split

train_val_idx, test_idx = train_test_split(
    data.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.25,
    random_state=42,
    stratify=y.loc[train_val_idx]
)

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_val, y_val = X.loc[val_idx], y.loc[val_idx]
X_test, y_test = X.loc[test_idx], y.loc[test_idx]

train_urls = set(data.loc[train_idx, "url"])
val_urls = set(data.loc[val_idx, "url"])
test_urls = set(data.loc[test_idx, "url"])

assert train_urls.isdisjoint(val_urls)
assert train_urls.isdisjoint(test_urls)
assert val_urls.isdisjoint(test_urls)

print("Training URLs:", len(X_train))
print("Validation URLs:", len(X_val))
print("Test URLs:", len(X_test))
print("\nNo exact URLs overlap between sets.")

Training URLs: 6857
Validation URLs: 2286
Test URLs: 2286

No exact URLs overlap between sets.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),

    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: training complete")

Baseline: training complete
Logistic Regression: training complete
Random Forest: training complete


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

results = []

for name, model in models.items():
    predictions = model.predict(X_val)

    phishing_column = list(model.classes_).index(1)
    scores = model.predict_proba(X_val)[:, phishing_column]

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "ROC AUC": roc_auc_score(y_val, scores),
        "False positive rate": fp / (fp + tn)
    })

validation_results = pd.DataFrame(results).sort_values(
    "F1", ascending=False
)
display(validation_results.round(4))

,Model,Accuracy,Precision,Recall,F1,ROC AUC,False positive rate
2,Random Forest,0.8495,0.8689,0.8233,0.8455,0.9293,0.1242
1,Logistic Regression,0.7787,0.8234,0.7095,0.7622,0.8692,0.1522
0,Baseline,0.5000,0.0000,0.0000,0.0000,0.5000,0.0000


In [ ]:
import numpy as np

selected_model = models["Random Forest"]

phishing_column = list(selected_model.classes_).index(1)
val_scores = selected_model.predict_proba(X_val)[:, phishing_column]
val_labels = y_val.to_numpy()

# Low risk: score < low_threshold
# Choose the largest cutoff, up to 0.5, that places
# at most 5% of validation phishing links in Low.
low_candidates = np.linspace(0, 0.5, 501)

valid_low = [
    threshold
    for threshold in low_candidates
    if np.mean(val_scores[val_labels == 1] < threshold) <= 0.05
]

low_threshold = max(valid_low)

# High risk: score >= high_threshold
# Choose the smallest cutoff, from 0.5 upward, that places
# at most 5% of validation legitimate links in High.
# The extra cutoff above 1 allows an empty High category if needed.
high_candidates = np.append(np.linspace(0.5, 1, 501), 1.001)

valid_high = [
    threshold
    for threshold in high_candidates
    if np.mean(val_scores[val_labels == 0] >= threshold) <= 0.05
]

high_threshold = min(valid_high)

def assign_risk(scores):
    scores = np.asarray(scores)
    return np.where(
        scores < low_threshold,
        "Low",
        np.where(scores >= high_threshold, "High", "Moderate")
    )

validation_risk = pd.DataFrame({
    "Risk category": assign_risk(val_scores),
    "Actual label": np.where(
        val_labels == 1, "Phishing", "Legitimate"
    )
})

risk_table = pd.crosstab(
    validation_risk["Risk category"],
    validation_risk["Actual label"]
).reindex(
    index=["Low", "Moderate", "High"],
    columns=["Legitimate", "Phishing"],
    fill_value=0
)

print(f"Low risk: score < {low_threshold:.3f}")
print(
    f"Moderate risk: {low_threshold:.3f} "
    f"<= score < {high_threshold:.3f}"
)
print(f"High risk: score >= {high_threshold:.3f}")
print("\nValidation results by risk category:")
display(risk_table)

print(
    "\nPhishing links assigned Low:",
    f"{np.mean(val_scores[val_labels == 1] < low_threshold):.2%}"
)
print(
    "Legitimate links assigned High:",
    f"{np.mean(val_scores[val_labels == 0] >= high_threshold):.2%}"
)

Low risk: score < 0.230
Moderate risk: 0.230 <= score < 0.697
High risk: score >= 0.697

Validation results by risk category:


Actual label,Legitimate,Phishing
Risk category,,
Low,756,57
Moderate,330,301
High,57,785



Phishing links assigned Low: 4.99%
Legitimate links assigned High: 4.99%


In [ ]:
from sklearn.metrics import classification_report

test_scores = selected_model.predict_proba(
    X_test
)[:, phishing_column]

test_predictions = selected_model.predict(X_test)
test_labels = y_test.to_numpy()

print("FINAL TEST RESULTS — default binary cutoff\n")

print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["Legitimate", "Phishing"],
    digits=4,
    zero_division=0
))

print("ROC AUC:", round(
    roc_auc_score(y_test, test_scores), 4
))

tn, fp, fn, tp = confusion_matrix(
    y_test, test_predictions, labels=[0, 1]
).ravel()

print(f"False positive rate: {fp / (fp + tn):.2%}")

test_risks = assign_risk(test_scores)

test_risk_results = pd.DataFrame({
    "Risk category": test_risks,
    "Actual label": np.where(
        test_labels == 1, "Phishing", "Legitimate"
    )
})

test_risk_table = pd.crosstab(
    test_risk_results["Risk category"],
    test_risk_results["Actual label"]
).reindex(
    index=["Low", "Moderate", "High"],
    columns=["Legitimate", "Phishing"],
    fill_value=0
)

print("\nFINAL TEST RESULTS — risk categories\n")
display(test_risk_table)

print(
    "Phishing links assigned Low:",
    f"{np.mean(test_risks[test_labels == 1] == 'Low'):.2%}"
)

print(
    "Legitimate links assigned High:",
    f"{np.mean(test_risks[test_labels == 0] == 'High'):.2%}"
)

print(
    "Links assigned Moderate:",
    f"{np.mean(test_risks == 'Moderate'):.2%}"
)

FINAL TEST RESULTS — default binary cutoff

              precision    recall  f1-score   support

  Legitimate     0.8528    0.8618    0.8573      1143
    Phishing     0.8603    0.8513    0.8558      1143

    accuracy                         0.8565      2286
   macro avg     0.8566    0.8565    0.8565      2286
weighted avg     0.8566    0.8565    0.8565      2286

ROC AUC: 0.9328
False positive rate: 13.82%

FINAL TEST RESULTS — risk categories



Actual label,Legitimate,Phishing
Risk category,,
Low,721,57
Moderate,366,278
High,56,808


Phishing links assigned Low: 4.99%
Legitimate links assigned High: 4.90%
Links assigned Moderate: 28.17%


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from urllib.parse import urlsplit

def evaluate_url(url):
    url = url.strip()

    if not url:
        raise ValueError("Please paste a URL.")

    if any(character.isspace() for character in url):
        raise ValueError("The URL cannot contain spaces or line breaks.")

    parse_url = url if "://" in url else "//" + url

    try:
        parts = urlsplit(parse_url)
        host = parts.hostname
        port = parts.port
    except ValueError:
        raise ValueError("This URL could not be parsed.")

    if parts.scheme.lower() not in ("", "http", "https"):
        raise ValueError("Please enter an HTTP or HTTPS website URL.")

    if not host or ("." not in host and ":" not in host):
        raise ValueError("Please enter a website address with a valid hostname.")

    features = extract_url_features(url)
    feature_row = pd.DataFrame([features]).reindex(columns=X.columns)

    class_column = list(selected_model.classes_).index(1)
    score = float(
        selected_model.predict_proba(feature_row)[0, class_column]
    )
    risk = assign_risk([score])[0]

    print(f"Estimated risk: {risk}")
    print(f"Model score: {score:.3f} (0–1 scale)")
    print("This score is not a verified probability of phishing.")

    messages = {
        "Low": "The model found lower risk. This does not establish safety.",
        "Moderate": "The result is uncertain. Verify the link independently.",
        "High": "The model flagged higher risk. Avoid entering sensitive information."
    }

    print("\n" + messages[risk])

    print("\nURL observations:")
    print(f"- Hostname: {host}")
    print(f"- URL length: {features['url_length']} characters")
    print(f"- Dots in hostname: {features['hostname_dot_count']}")
    print(f"- HTTPS explicitly present: {'Yes' if features['uses_https'] else 'No'}")
    print(f"- IP address as hostname: {'Yes' if features['host_is_ip'] else 'No'}")
    print(f"- @ characters: {features['count_at_signs']}")


url_input = widgets.Text(
    placeholder="Paste a website URL here",
    description="URL:",
    layout=widgets.Layout(width="95%")
)

check_button = widgets.Button(
    description="Evaluate link",
    button_style="primary"
)

result_output = widgets.Output()

def on_check_clicked(button):
    with result_output:
        clear_output(wait=True)
        try:
            evaluate_url(url_input.value)
        except ValueError as error:
            print(error)

check_button.on_click(on_check_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Phishing URL Risk Checker</h3>"),
    url_input,
    check_button,
    result_output
]))